# Week 7 - mainly webscraping with a big recap on API's
- recap api's
- requests
- adding regex
- webscraping with BeautifulSoup
- downloading images?
- combining api and webscraping

In [ ]:
# https://techy-api.vercel.app/api/json

import requests, json
url = "https://techy-api.vercel.app/api/json"
response = requests.get(url)

#response = requests.request("GET", url) --> old way of doing things
if response.status_code == 200:
    data = json.loads(response.text)
    print(f'Quote of the day: \n{data["message"]}')
else:
    print("No access, try again")

Quote of the day: 
It was very hard doing the secondary modulation


In [10]:
# find out in which movies baloo the bear plays a role
import requests, json

url = "https://api.disneyapi.dev/character"
response = requests.get(url)

if response.status_code == 200:
    data = json.loads(response.text)
    for character in data["data"]:
        if character["name"] == "Baloo":
            listOfFilms = character["films"]
else:
    print("try again, something failed")

print("Baloo stars in the following films:")
for film in listOfFilms:
    print(film)


Baloo stars in the following films:
The Jungle Book
The Jungle Book 2
Mickey's Magical Christmas: Snowed in at the House of Mouse
Mickey's House of Villains


In [ ]:
#find the character in the api that has the most media --> films, shortfilms and tvshows combined.
import requests, json

url = "https://api.disneyapi.dev/character"
response = requests.get(url)

counter = 0

if response.status_code == 200:
    data = json.loads(response.text)
    for character in data["data"]:
        amount = len(character["films"]) + len(character["shortFilms"]) + len(character["tvShows"])
        if amount > counter:
            counter = amount
            amount = 0
            name = character["name"]
    print(f"{name} has the most media --> {counter}")
else:
    print("try again, something failed")

#return a list of characters (name) that have at least 1 videogame and 1 film.
listofchars = []
if response.status_code == 200:
    data = json.loads(response.text)
    for character in data["data"]:
        if len(character["videoGames"]) > 0 and len(character["films"]) > 0:
            listofchars.append(character["name"])
    print(listofchars)
else:
    print("try again, something failed")

#other options --> character["videoGames"] != [] and character["films"] --> option 3: character["films"] is not Null

Baloo has the most media --> 8
['Achilles', 'Baloo', 'Beheaded Knight', 'Captain Amelia']


## Webscraping

In [16]:
import requests, re
url = "https://en.wikipedia.org/wiki/JavaScript"
headers = {'User-Agent' : 'Anthony'}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    html = response.text
    regex = "object"
    matches = re.findall(regex, html.lower())
    print(len(matches))
else:
    print("Try again, probably not access or something wrong")

144


## Working with BeautifulSoup

In [5]:
! python -m pip install beautifulsoup4


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Amount of word: object inside of the big soup of html
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/JavaScript"
headers = {'User-Agent' : 'Anthony'}
response = requests.get(url, headers=headers)

html = response.text
soup = BeautifulSoup(html)
print(soup.title)
print(soup.title.get_text())
print(soup.get_text().lower().count("object"))
print(soup.get_text()[0:500:].split())

h3_tags = soup.find("h3")
print(h3_tags)

h3tags = soup.find_all("h3")
for tag in h3tags:
    print(tag.get_text())


<title>JavaScript - Wikipedia</title>
JavaScript - Wikipedia
76
['JavaScript', '-', 'Wikipedia', 'Jump', 'to', 'content', 'Main', 'menu', 'Main', 'menu', 'move', 'to', 'sidebar', 'hide', 'Navigation', 'Main', 'pageContentsCurrent', 'eventsRandom', 'articleAbout', 'WikipediaContact', 'us', 'Contribute', 'HelpLearn', 'to', 'editCommunity', 'portalRecent', 'changesUpload', 'fileSpecial', 'pages', 'Search', 'Search', 'Appearance', 'Donate', 'Create', 'account', 'Log', 'in', 'Personal', 'tools', 'Donate', 'Create', 'account', 'Log', 'in']
<h3 id="Creation_at_Netscape">Creation at Netscape</h3>
Creation at Netscape
Adoption by Microsoft
The rise of JScript
Growth and standardization
Reaching maturity
Examples of scripted behavior
Libraries and frameworks
JavaScript engine
Runtime system
Imperative and structured
Weakly typed
Dynamic
Object-orientation (prototype-based)
Functional
Delegative
Miscellaneous
Vendor-specific extensions
Cross-site scripting
Cross-site request forgery
Misplaced tru

In [36]:
#finding elephant in different languages
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Elephant"
headers = {'User-Agent' : 'Anthony'}
response = requests.get(url, headers=headers)

html = response.text
soup = BeautifulSoup(html)
liTags = soup.find_all("li", {'class':'interlanguage-link'})

language = input()
for tag in liTags:
    if language == tag.a["title"].split()[-1]:
        word = tag.a["title"].split()[0]
        print(f"Elephant in {language} = {word}")

In [ ]:
#downloading all images from a webpage
import requests, re
from bs4 import BeautifulSoup
import time

url = "https://en.wikipedia.org/wiki/Film"
headers = {'User-Agent': 'Anthony'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text)

imgTags = soup.find_all("img")
sources = [img["src"] for img in imgTags]
print(sources)
updatedSources = []
for source in sources:
    if re.search(r"(\.(jpg|png|gif))", source):
        updatedSources.append(source)

    
for img in updatedSources:
    url = "https:" + img
    r = requests.get(url, headers=headers, allow_redirects=True)
    print(r.status_code)
    time.sleep(5)
    with open("Images\\" + img.split("/")[-2], "wb") as file:
        file.write(r.content)

['/static/images/icons/enwiki-25.svg', '/static/images/mobile/copyright/wikipedia-wordmark-en-25.svg', '/static/images/mobile/copyright/wikipedia-tagline-en-25.svg', '//upload.wikimedia.org/wikipedia/en/thumb/1/1b/Semi-protection-shackle.svg/20px-Semi-protection-shackle.svg.png', '//upload.wikimedia.org/wikipedia/en/thumb/9/99/Question_book-new.svg/60px-Question_book-new.svg.png', '//upload.wikimedia.org/wikipedia/en/thumb/e/e7/Video-x-generic.svg/120px-Video-x-generic.svg.png', '//upload.wikimedia.org/wikipedia/en/thumb/e/e7/Video-x-generic.svg/20px-Video-x-generic.svg.png', '//upload.wikimedia.org/wikipedia/commons/thumb/b/bf/Prof._Stampfer%27s_Stroboscopische_Scheibe_No._X.gif/250px-Prof._Stampfer%27s_Stroboscopische_Scheibe_No._X.gif', '//upload.wikimedia.org/wikipedia/commons/thumb/0/07/The_Horse_in_Motion-anim.gif/250px-The_Horse_in_Motion-anim.gif', '//upload.wikimedia.org/wikipedia/commons/thumb/e/e8/Electrotachyscope1.jpg/250px-Electrotachyscope1.jpg', '//upload.wikimedia.org/